# 02 Silver transform
Parses the raw Bronze JSON into typed, validated tables.
**Reads:** `bronze_scoreboard`, `bronze_game_summary`  
**Writes:** `silver_games`, `silver_player_box_scores`

In [0]:
# --- Setup for the Silver notebook ---
import os, sys
from pyspark.sql import functions as F   # Spark's built-in column functions

# Make our src/ package importable (the notebook lives in notebooks/, so go up one level)
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(repo_root, "src"))

# The quality gate we just wrote
from nba_pipeline.quality import enforce_checks, DataQualityError
print("ready")

## Part 1: silver_games
One row per final game. Scores are cast from strings to integers, and
`game_date` comes from the requested date because UTC tip-off times can cross midnight.

In [0]:
# --- Build the games DataFrame from bronze_scoreboard (nothing is saved yet) ---
from pyspark.sql import functions as F

# 1. The schema: only the fields we want. Everything else in the JSON is ignored.
#    Struct = JSON object, ARRAY = JSON list.
scoreboard_schema = """
events ARRAY<STRUCT<
    id: STRING,
    date: STRING,
    name: STRING,
    status: STRUCT<type: STRUCT<name: STRING>>,
    competitions: ARRAY<STRUCT<
        neutralSite: BOOLEAN,
        competitors: ARRAY<STRUCT<
            homeAway: STRING,
            score: STRING,
            team: STRUCT<id: STRING, abbreviation: STRING, displayName: STRING>
        >>
    >>
>>
"""

# 2. Parse the raw JSON text into a struct column named "j" (runs in parallel)
parsed = spark.table("workspace.nba.bronze_scoreboard").select(
    "source_file",
    F.from_json("raw_json", scoreboard_schema).alias("j"),
)

# 3. explode: turn the array of games in each file into one row per game
events = parsed.select("source_file", F.explode("j.events").alias("e"))

# 4. Pull out the home and away team structs. filter() with a lambda scans the
#    competitors array for the matching team, and [0] takes the first match.
games = events.select(
    "source_file",
    "e",
    F.expr("filter(e.competitions[0].competitors, c -> c.homeAway = 'home')[0]").alias("home"),
    F.expr("filter(e.competitions[0].competitors, c -> c.homeAway = 'away')[0]").alias("away"),
)

# 5. Final column selection: flat names, proper types
silver_games_df = games.select(
    F.col("e.id").alias("game_id"),
    # The date we requested from the API, taken from the Bronze file name
    F.to_date(F.regexp_extract("source_file", r"game_date=(\d{4}-\d{2}-\d{2})", 1)).alias("game_date"),
    # Tip-off time, parsed from text like "2026-10-20T19:00Z"
    F.to_timestamp("e.date", "yyyy-MM-dd'T'HH:mm'Z'").alias("tipoff_utc"),
    F.col("e.name").alias("matchup"),
    F.col("e.status.type.name").alias("status"),
    F.col("e.competitions")[0]["neutralSite"].alias("neutral_site"),
    F.col("home.team.id").alias("home_team_id"),
    F.col("home.team.abbreviation").alias("home_team_abbr"),
    F.col("home.team.displayName").alias("home_team_name"),
    F.col("home.score").cast("int").alias("home_score"),   # string -> integer
    F.col("away.team.id").alias("away_team_id"),
    F.col("away.team.abbreviation").alias("away_team_abbr"),
    F.col("away.team.displayName").alias("away_team_name"),
    F.col("away.score").cast("int").alias("away_score"),
    "source_file",
)

# 6. Inspect: schema first (cheap), then actions that trigger the real work
silver_games_df.printSchema()
print(silver_games_df.count(), "rows")
print(silver_games_df.select("game_id").distinct().count(), "distinct game_ids")
silver_games_df.drop("source_file").show(5, truncate=40)

In [0]:
# --- Data quality checks on silver_games_df (nothing is saved) ---
from pyspark.sql import functions as F

# Helper: counts the rows that BREAK a rule (so 0 means the check passes)
def bad(condition):
    return F.sum(F.when(condition, 1).otherwise(0))

# 1. All row-level checks in ONE aggregation (a single pass over the data)
agg_row = silver_games_df.agg(
    F.count("*").alias("total_rows"),
    # Missing identifiers
    bad(F.col("game_id").isNull() | F.col("home_team_id").isNull() | F.col("away_team_id").isNull()).alias("null_ids"),
    # Missing scores or tip-off time (null comparisons aren't caught by the rules below)
    bad(F.col("home_score").isNull() | F.col("away_score").isNull()).alias("null_scores"),
    bad(F.col("tipoff_utc").isNull()).alias("null_tipoff"),
    # NBA games always have points scored by both teams
    bad((F.col("home_score") <= 0) | (F.col("away_score") <= 0)).alias("non_positive_scores"),
    # NBA games can't end in a tie
    bad(F.col("home_score") == F.col("away_score")).alias("ties"),
    # A team can't play itself
    bad(F.col("home_team_id") == F.col("away_team_id")).alias("same_team"),
    # We only fetched final games
    bad(F.col("status") != "STATUS_FINAL").alias("not_final"),
    # Dates should fall inside the range we backfilled
    bad((F.col("game_date") < F.lit("2026-10-20").cast("date")) |
        (F.col("game_date") > F.lit("2026-11-15").cast("date"))).alias("date_out_of_range"),
    # The UTC tip-off date should be the game date or the day after (late US games)
    bad(~F.datediff(F.to_date("tipoff_utc"), F.col("game_date")).isin(0, 1)).alias("tipoff_date_mismatch"),
).first()
# Stop with a clear message if no row came back; this also satisfies the editor's type checker.
assert agg_row is not None, "Quality check aggregation returned no row"
# Convert the Row into a plain dict of {check name: count}.
results = agg_row.asDict()

# 2. Cross-row check: no team should play twice on the same date.
#    Stack home and away teams into one list, then count games per team per date.
team_games = (
    silver_games_df.select("game_date", F.col("home_team_id").alias("team_id"))
    .unionByName(silver_games_df.select("game_date", F.col("away_team_id").alias("team_id")))
)
results["team_twice_same_day"] = (
    team_games.groupBy("game_date", "team_id").count().where("count > 1").count()
)

# 3. Gate: print PASS/FAIL per check, then RAISE if any check failed.
#    Raising stops the notebook here, so the silver_games write cell below never runs.
total = results.pop("total_rows")
enforce_checks(results, total, "silver_games")

In [0]:
# --- Write silver_games as a Delta table ---
# Relies on: silver_games_df

SILVER_GAMES = "workspace.nba.silver_games"   # catalog.schema.table

# The ACTION: Spark now runs the whole plan (read Bronze -> parse -> flatten -> cast)
# and stores the result. overwrite makes the cell safe to re-run.
(
    silver_games_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_GAMES)
)

# Add a description that shows up in the Catalog UI
spark.sql(f"COMMENT ON TABLE {SILVER_GAMES} IS 'One row per final NBA game: date, teams, and scores. Parsed and typed from bronze_scoreboard.'")

# Verify 1: read the saved table back and count its rows
print(spark.table(SILVER_GAMES).count(), "rows in", SILVER_GAMES)

# Verify 2: the transaction log entry for this write, with its metrics
display(
    spark.sql(f"DESCRIBE HISTORY {SILVER_GAMES}")
    .select("version", "operation", "operationMetrics")
)

## Part 2: silver_player_box_scores
One row per player per game. ESPN sends stats as a positional list of strings,
so values are mapped by position (verified identical across all games).
Runs after Part 1 because the reconciliation check reads `silver_games`.

In [0]:
# --- Build the player box score DataFrame from bronze_game_summary (nothing is saved) ---
from pyspark.sql import functions as F

# 1. Schema: only the fields we need. Everything else in the huge JSON is ignored.
box_schema = """
boxscore STRUCT<
    players: ARRAY<STRUCT<
        team: STRUCT<id: STRING, abbreviation: STRING, displayName: STRING>,
        statistics: ARRAY<STRUCT<
            labels: ARRAY<STRING>,
            athletes: ARRAY<STRUCT<
                athlete: STRUCT<id: STRING, displayName: STRING, jersey: STRING,
                                position: STRUCT<abbreviation: STRING>>,
                starter: BOOLEAN,
                didNotPlay: BOOLEAN,
                stats: ARRAY<STRING>
            >>
        >>
    >>
>
"""

# 2. Parse the raw JSON text into a struct column "j" (runs in parallel)
parsed = spark.table("workspace.nba.bronze_game_summary").select(
    "source_file",
    F.from_json("raw_json", box_schema).alias("j"),
)

# 3. First explode: one row per TEAM per game ("tb" = team block)
teams = parsed.select("source_file", F.explode("j.boxscore.players").alias("tb"))

# 4. Second explode: one row per PLAYER. try_element_at(..., 1) takes the first
#    statistics block and returns null (instead of failing) if it's missing.
players = teams.select(
    "source_file",
    F.col("tb.team").alias("team"),
    F.explode(F.expr("try_element_at(tb.statistics, 1).athletes")).alias("a"),
)

# 5. Helpers that build SQL expressions over the positional stats list "a.stats".
#    Positions are 1-based and follow the labels:
#    MIN PTS FG 3PT FT REB AST TO STL BLK OREB DREB PF +/-
def stat_int(pos):
    # Take the stat at this position and cast to int (null if missing or not numeric)
    return F.expr(f"try_cast(try_element_at(a.stats, {pos}) as int)")

def made(pos):
    # '12-21' -> split on '-' -> first part (made) as int
    return F.expr(f"try_cast(try_element_at(split(try_element_at(a.stats, {pos}), '-'), 1) as int)")

def attempted(pos):
    # '12-21' -> split on '-' -> second part (attempted) as int
    return F.expr(f"try_cast(try_element_at(split(try_element_at(a.stats, {pos}), '-'), 2) as int)")

# 6. Final columns: flat names and proper types
silver_box_df = players.select(
    F.regexp_extract("source_file", r"game_id=(\d+)", 1).alias("game_id"),
    F.col("team.id").alias("team_id"),
    F.col("team.abbreviation").alias("team_abbr"),
    F.col("a.athlete.id").alias("player_id"),
    F.col("a.athlete.displayName").alias("player_name"),
    F.col("a.athlete.position.abbreviation").alias("position"),
    F.col("a.athlete.jersey").alias("jersey"),
    F.col("a.starter").alias("starter"),
        # A player counts as "did not play" if ESPN flags them (didNotPlay = true) OR if
    # ESPN sends "--" as their minutes. A known quirk: some players are marked as
    # playing (didNotPlay = false) but have "--" minutes and all-zero stats.
    # coalesce(..., False) matters: a comparison against a missing value is null,
    # and "False OR null" would otherwise leave did_not_play null.
    (
        F.coalesce(F.col("a.didNotPlay"), F.lit(False))
        | F.coalesce(F.expr("try_element_at(a.stats, 1) = '--'"), F.lit(False))
    ).alias("did_not_play"),
    stat_int(1).alias("minutes"),
    stat_int(2).alias("points"),
    made(3).alias("fg_made"),       attempted(3).alias("fg_attempted"),
    made(4).alias("fg3_made"),      attempted(4).alias("fg3_attempted"),
    made(5).alias("ft_made"),       attempted(5).alias("ft_attempted"),
    stat_int(6).alias("rebounds"),
    stat_int(7).alias("assists"),
    stat_int(8).alias("turnovers"),
    stat_int(9).alias("steals"),
    stat_int(10).alias("blocks"),
    stat_int(11).alias("off_rebounds"),
    stat_int(12).alias("def_rebounds"),
    stat_int(13).alias("fouls"),
    stat_int(14).alias("plus_minus"),
    "source_file",
)

# 7. Inspect: schema first (cheap), then actions that trigger the real work
silver_box_df.printSchema()
print(silver_box_df.count(), "player rows")
print(silver_box_df.select("game_id").distinct().count(), "distinct games")

# Rows split by played / did not play
silver_box_df.groupBy("did_not_play").count().show()

# 8. Confirm the 14 stat labels are identical in EVERY game (the positional
#    mapping above depends on it). We expect exactly 1 distinct label list.
label_lists = teams.select(
    F.expr("try_element_at(tb.statistics, 1).labels").alias("labels")
).distinct()
print(label_lists.count(), "distinct label list(s) across all games")
label_lists.show(truncate=120)

# 9. Look at a few rows: a starter, plus a DNP player to see the nulls
silver_box_df.drop("source_file").where("game_id = '401909088'") \
    .orderBy(F.col("team_abbr"), F.col("starter").desc()).show(8, truncate=20)

In [0]:
# --- Data quality checks on silver_box_df (nothing is saved) ---
# Relies on: silver_box_df and the silver_games table.
from functools import reduce
from pyspark.sql import functions as F

# Helper: counts the rows that BREAK a rule (so 0 means the check passes)
def bad(condition):
    return F.sum(F.when(condition, 1).otherwise(0))

# Handy shorthand: True for players who actually played
played = ~F.col("did_not_play")

# Column groups used by the checks
count_cols = ["minutes", "points", "fg_made", "fg_attempted", "fg3_made", "fg3_attempted",
              "ft_made", "ft_attempted", "rebounds", "assists", "turnovers", "steals",
              "blocks", "off_rebounds", "def_rebounds", "fouls"]   # can never be negative
all_stat_cols = count_cols + ["plus_minus"]                          # plus_minus can be negative

# Helper: combine a list of conditions with OR (any column breaks the rule)
def any_of(conds):
    return reduce(lambda a, b: a | b, conds)

# 1. All row-level checks in ONE aggregation (a single pass over the data)
agg_row = silver_box_df.agg(
    F.count("*").alias("total_rows"),
    # Missing identifiers
    bad(F.col("game_id").isNull() | F.col("team_id").isNull() | F.col("player_id").isNull()).alias("null_keys"),
    # Players who played must have every stat filled in
    bad(played & any_of([F.col(c).isNull() for c in all_stat_cols])).alias("played_with_null_stats"),
    # Players who did not play must have no stats
    bad(F.col("did_not_play") & F.col("minutes").isNotNull()).alias("dnp_with_stats"),
    # Counting stats can't be negative
    bad(played & any_of([F.col(c) < 0 for c in count_cols])).alias("negative_stats"),
    # You can't make more shots than you attempt
    bad(played & ((F.col("fg_made") > F.col("fg_attempted")) |
                  (F.col("fg3_made") > F.col("fg3_attempted")) |
                  (F.col("ft_made") > F.col("ft_attempted")))).alias("made_gt_attempted"),
    # Threes are a subset of field goals
    bad(played & ((F.col("fg3_made") > F.col("fg_made")) |
                  (F.col("fg3_attempted") > F.col("fg_attempted")))).alias("threes_exceed_fg"),
    # Points must follow from the shooting numbers
    bad(played & (F.col("points") != 2 * F.col("fg_made") + F.col("fg3_made") + F.col("ft_made"))).alias("points_formula_mismatch"),
    # Total rebounds = offensive + defensive
    bad(played & (F.col("rebounds") != F.col("off_rebounds") + F.col("def_rebounds"))).alias("rebound_mismatch"),
    # 48 minutes plus up to 5 overtimes is far below 70
    bad(played & (F.col("minutes") > 70)).alias("minutes_over_70"),
).first()
# Stop with a clear message if no row came back; this also satisfies the editor's type checker.
assert agg_row is not None, "Quality check aggregation returned no row"
# Convert the Row into a plain dict of {check name: count}.
results = agg_row.asDict()

# 2. Cross-row check: no player should appear twice in the same game
results["duplicate_player_rows"] = (
    silver_box_df.groupBy("game_id", "player_id").count().where("count > 1").count()
)

# 3. Cross-row check: every game should have exactly 2 teams
results["games_without_2_teams"] = (
    silver_box_df.groupBy("game_id").agg(F.countDistinct("team_id").alias("n_teams"))
    .where("n_teams != 2").count()
)

# 4. Cross-TABLE reconciliation: player points per team vs the final score
#    in silver_games. Step A: sum player points per team per game.
player_points = (
    silver_box_df.groupBy("game_id", "team_id")
    .agg(F.sum("points").alias("player_points"))
)

#    Step B: stack the home and away scores into one "team score" list.
games_tbl = spark.table("workspace.nba.silver_games")
team_scores = (
    games_tbl.select("game_id", F.col("home_team_id").alias("team_id"), F.col("home_score").alias("team_score"))
    .unionByName(games_tbl.select("game_id", F.col("away_team_id").alias("team_id"), F.col("away_score").alias("team_score")))
)

#    Step C: FULL OUTER join, so a team or game missing on either side still shows up.
#    Keep only the rows that disagree (or have a null on one side).
mismatch = (
    player_points.join(team_scores, ["game_id", "team_id"], "full_outer")
    .where(F.col("player_points").isNull() | F.col("team_score").isNull()
           | (F.col("player_points") != F.col("team_score")))
)
results["team_points_vs_score_mismatch"] = mismatch.count()

# 5. Pull the row count out of the results dictionary
total = results.pop("total_rows")

# 6. If team points failed to reconcile, show the offending team-games BEFORE the gate.
#    The gate raises an exception, which ends the cell, so anything after it wouldn't run.
if results["team_points_vs_score_mismatch"] > 0:
    mismatch.show(10)

# 7. Gate: print PASS/FAIL per check, then RAISE if any check failed.
enforce_checks(results, total, "silver_player_box_scores")

In [0]:
# --- Write silver_player_box_scores as a Delta table ---
# Relies on: silver_box_df

SILVER_BOX = "workspace.nba.silver_player_box_scores"   # catalog.schema.table

# The ACTION: Spark runs the whole plan (read Bronze -> parse -> explode twice -> cast)
# and stores the result. overwrite makes the cell safe to re-run.
(
    silver_box_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_BOX)
)

# Add a description that shows up in the Catalog UI
spark.sql(f"COMMENT ON TABLE {SILVER_BOX} IS 'One row per player per final NBA game: minutes and box score stats. Parsed and typed from bronze_game_summary. Players with did_not_play = true have null stats.'")

# Verify 1: read the saved table back and count its rows
print(spark.table(SILVER_BOX).count(), "rows in", SILVER_BOX)

# Verify 2: a quick sanity query on the saved table.
# The five highest scoring performances so far this season.
spark.sql(f"""
    SELECT game_id, team_abbr, player_name, points, minutes
    FROM {SILVER_BOX}
    ORDER BY points DESC
    LIMIT 5
""").show()

# Time travel demo on the Bronze table: it has two versions because its write
# cell ran twice. Both versions should hold the same 197 rows (overwrite, not append).
for v in (0, 1):
    n = spark.sql(f"SELECT count(*) AS n FROM workspace.nba.bronze_game_summary VERSION AS OF {v}").first()["n"]
    print(f"bronze_game_summary version {v}: {n} rows")